In [1]:
import os
import sys
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.stats import gamma, zscore, pearsonr
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import GroupKFold
import nibabel as nib
from pathlib import Path
import warnings
os.environ['PATH'] = '/opt/workbench/bin_linux64:' + os.environ['PATH']

# Suppress warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def preprocess_fmri(fmri_data, timing_df, TR, bin_sec, delay_sec, drop_last=True):
    """
    Extract and bin fMRI data with BOLD delay compensation.
    """
    n_vertices, n_timepoints = fmri_data.shape
    segments = []
    
    for _, row in timing_df.iterrows():
        # --- FIX: Removed trailing commas below ---
        onset_sec = row["onset_sec"]
        duration_sec = row["duration_sec"]

        # Apply delay to the stimulus window
        onset_d = onset_sec + delay_sec
        end_d = onset_sec + duration_sec + delay_sec

        # Convert to timepoint indices
        start_tp = int(np.floor(onset_d / TR)) 
        end_tp = int(np.ceil(end_d / TR))

        # Clip to valid range
        start_tp = max(start_tp, 0)
        end_tp = min(end_tp, n_timepoints)

        # Handle edge case where delay pushes segment out of bounds
        if end_tp <= start_tp:
            raise RuntimeError(f'Segmentation error from {onset_sec} to {onset_sec+duration_sec}') 

        segment = fmri_data[:, start_tp:end_tp]

        # Timepoints per bin
        tp_per_bin = int(np.floor(bin_sec / TR))
        if tp_per_bin < 1:
            raise ValueError(f"bin_sec/TR is < 1 (bin_sec={bin_sec}, TR={TR})")

        # Drop leftover timepoints that don't fill a full bin
        if segment.shape[1] == 0:
            continue
            
        if drop_last:
            n_bins = int(np.floor(segment.shape[1] / tp_per_bin))
        else:
            n_bins = int(np.ceil(segment.shape[1] / tp_per_bin))
            
        # Ensure we have enough data for at least one bin
        if n_bins == 0:
            continue
            
        # Truncate to exact multiple of bin size to allow reshape
        # If not dropping last, we might need padding, but standard approach here 
        # usually implies we only take what fits or we strictly reshape.
        # Given your code used reshape(), we must ensure exact size.
        valid_length = n_bins * tp_per_bin
        
        # If strict floor was used, this just trims tail
        if segment.shape[1] >= valid_length:
            segment = segment[:, :valid_length]
        else:
            # If we calculated bins via ceil but don't have enough data (rare), skip
            continue

        binned = segment.reshape(n_vertices, n_bins, tp_per_bin).mean(axis=2)
        segments.append(binned)
        # print(f"  Video {row['video_id']}: {binned.shape[1]} bins")
    
    if len(segments) == 0:
        return np.zeros((n_vertices, 0), dtype=fmri_data.dtype)
        
    filtered_fmri = np.concatenate(segments, axis=1)
    return filtered_fmri

def spm_hrf(tr, oversampling=16):
    dt = tr / oversampling
    u = np.arange(0, 32, dt) - dt/2
    h = (6 * gamma.pdf(u, 6, loc=0, scale=1) - 1/6 * gamma.pdf(u, 16, loc=0, scale=1))
    return h / np.sum(h)

def process_model_embeddings(model_embeddings, bin_sec, timing_df, hrf=True, normalization=True):
    """
    Handles HRF convolution and PER-RUN normalization.
    """
    # 1. Setup HRF
    hrf_kernel = spm_hrf(bin_sec, oversampling=1)
    
    # 2. Slice into individual video segments
    video_bin_counts = (timing_df['duration_sec'] / bin_sec).astype(int).values
    segments = []
    current_idx = 0
    
    for n_bins in video_bin_counts:
        segment = model_embeddings[current_idx : current_idx + n_bins, :]
        
        # Apply HRF if requested
        if hrf:
            conv_segment = np.zeros_like(segment)
            for i in range(segment.shape[1]):
                # Convolve and trim tail to maintain video length
                conv = np.convolve(segment[:, i], hrf_kernel, mode='full')
                conv_segment[:, i] = conv[:n_bins]
            segments.append(conv_segment)
        else:
            segments.append(segment)
            
        current_idx += n_bins

    # 3. Perform Per-Run Normalization
    if normalization:
        print("Performing Per-Run Normalization...")
        # Create a placeholder list for normalized segments
        normalized_segments = [None] * len(segments)
        unique_runs = timing_df['run_id'].unique()
        
        for rid in unique_runs:
            # Find indices of videos belonging to this run
            run_indices = timing_df.index[timing_df['run_id'] == rid].tolist()
            
            # Concatenate all segments in this run
            run_data = np.concatenate([segments[i] for i in run_indices], axis=0)
            
            # Z-score the entire run block
            run_z = zscore(run_data, axis=0)
            run_z = np.nan_to_num(run_z) # Clean up constant features
            
            # Slice the Z-scored run block back into original segments
            slice_ptr = 0
            for i in run_indices:
                v_len = video_bin_counts[i]
                normalized_segments[i] = run_z[slice_ptr : slice_ptr + v_len, :]
                slice_ptr += v_len
        
        segments = normalized_segments

    # 4. Re-concatenate into final embedding matrix
    return np.concatenate(segments, axis=0)

def run_ridge_cv(X, Y, groups, alphas, chunk_size):
    """
    Performs Ridge Regression with Leave-One-Run-Out Cross-Validation.
    Returns the correlation between predicted and actual brain activity 
    on held-out runs.
    """
    Y_T = Y.T  # Shape: (Samples, Vertices)
    n_samples, n_features = X.shape
    n_v = Y_T.shape[1]
    
    # Storage for out-of-sample predictions
    final_predictions = np.zeros((n_samples, n_v))
    
    # Initialize the splitter (Splits by Run ID)
    gkf = GroupKFold(n_splits=len(np.unique(groups)))
    
    # Loop through vertices in chunks to save memory
    for i in range(0, n_v, chunk_size):
        slc = slice(i, min(i + chunk_size, n_v))
        y_chunk = Y_T[:, slc]
        
        # Identify vertices that are not just zeros/noise
        valid_mask = np.std(y_chunk, axis=0) > 1e-10
        if not np.any(valid_mask):
            print(f"Progress: {min(i+chunk_size, n_v)}/{n_v}", end='\r')
            continue
            
        y_valid = y_chunk[:, valid_mask]
        preds_chunk = np.zeros_like(y_valid)
        
        # --- OUTER CV LOOP (Evaluation) ---
        for train_idx, test_idx in gkf.split(X, y_valid, groups):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y_valid[train_idx], y_valid[test_idx]
            
            # --- INNER CV LOOP (Selection) ---
            # RidgeCV performs efficient LOOCV internally on X_train to pick alpha
            model = RidgeCV(alphas=alphas) 
            model.fit(X_train, y_train)
            
            # Predict on held-out run
            preds_chunk[test_idx] = model.predict(X_test)
        # -----------------------------

        # Store predictions
        final_predictions[:, slice(i, i+chunk_size)][:, valid_mask] = preds_chunk
        print(f"Progress: {min(i+chunk_size, n_v)}/{n_v}", end='\r')
        
    print("\nCalculating correlations...")
    
    # Calculate correlation between Actual (Y_T) and Predicted (final_predictions)
    corrs = np.zeros(n_v)
    
    Y_centered = Y_T - Y_T.mean(axis=0)
    P_centered = final_predictions - final_predictions.mean(axis=0)
    
    with np.errstate(divide='ignore', invalid='ignore'):
        num = (Y_centered * P_centered).sum(axis=0)
        den = np.sqrt((Y_centered**2).sum(axis=0) * (P_centered**2).sum(axis=0))
        corrs = num / den
        corrs[np.isnan(corrs)] = 0 
        
    return corrs

def save_gifti(data, path):
    da = nib.gifti.GiftiDataArray(data.astype(np.float32), intent='NIFTI_INTENT_NONE', datatype='NIFTI_TYPE_FLOAT32')
    nib.save(nib.GiftiImage(darrays=[da]), str(path))

In [4]:
# ============================================================================
#  CONFIGURATION
# ============================================================================
''' 
Based on viewing the results, this configuration was chosen. de-meaning was added to when normalization is not performed. as a result the un-normalized results 
of configs other than 5s delay which didnt use normalization are not the latest version of the code 
(previous version used nothing when normalization is false; current one does demeaning). so the the only cofigurations with the latest variations in code are the ones
who are saved as cifti.
'''
use_normalization = [True, False]
use_hrf = [True , False]
BIN_SEC_LIST = [1.0, 2.0, 5.0, 10.0]  
DELAY_SEC_LIST = [ 5.0]

# Paths
template_cifti_path = '/home/amin/Research/Representation/Movie/data/Setareh/HCP_S1200_GroupAvg_v1/S1200.curvature_MSMAll.32k_fs_LR.dscalar.nii'
EMBEDDINGS_BASE = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings")
MAT_FILE_LEFT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_left_Meanfile.mat"
MAT_FILE_RIGHT = "/home/amin/Research/Representation/Movie/data/Setareh/HCP Data/notmean_right_Meanfile.mat"
TIMING_CSV = "/home/amin/Research/Representation/Movie/data/Setareh/Data/movie_timing.csv"
OUTPUT_BASE = Path("/home/amin/Research/Representation/Movie/outputs/encoding_results") 

# Models & Modalities
models = {"pe-av-small-16-frame"}
modalities = {"joint": "av",
            #    "audio": "a", "video": "v"
               }
TR = 1.0  # HCP 7T Specific

# Ridge Parameters
ALPHAS = [0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]
CHUNK_SIZE = 2000

# ============================================================================
#  EXECUTION
# ============================================================================
print("STEP 0: LOADING RAW DATA")
timing_df = pd.read_csv(TIMING_CSV)

fmri_l_raw = sio.loadmat(MAT_FILE_LEFT)['Left_data']
fmri_r_raw = sio.loadmat(MAT_FILE_RIGHT)['Right_data']
valid_mask_left = fmri_l_raw.std(axis=1) > 0
valid_mask_right = fmri_r_raw.std(axis=1) > 0
if fmri_l_raw.shape[0] < fmri_l_raw.shape[1]:
    print("WARNING: Dimensions look inverted (Time x Vertices). Transposing to (Vertices x Time).")
    fmri_l_raw = fmri_l_raw.T
    fmri_r_raw = fmri_r_raw.T

print(f"  Left Data Shape: {fmri_l_raw.shape} (Expect: Vertices x Time)")
print(f"  Right Data Shape: {fmri_r_raw.shape} (Expect: Vertices x Time)")


for hrf in use_hrf:
    current_delays = [0] if hrf else DELAY_SEC_LIST
    for delay_sec in current_delays:
        for bin_sec in BIN_SEC_LIST:
            # 1. Preprocess fMRI (Binning & Delay)
            print(f"\n[FMRI] Prep: Bin={bin_sec}s, Delay={delay_sec}s")
            
            # Using your new preprocess_fmri function
            fmri_l_base = preprocess_fmri(fmri_l_raw[valid_mask_left], timing_df, TR, bin_sec, delay_sec)
            fmri_r_base = preprocess_fmri(fmri_r_raw[valid_mask_right], timing_df, TR, bin_sec, delay_sec)
            
            unique_runs = timing_df['run_id'].unique()
            video_bins = (timing_df['duration_sec'] / bin_sec).astype(int).values
            # 2. Build Group Labels (Run IDs) for Cross-Validation
            # We must create a list of run_ids that matches the length of the binned data
            group_list = []
            video_bin_counts = (timing_df['duration_sec'] / bin_sec).astype(int).values
            run_ids = timing_df['run_id'].values

            for idx, count in enumerate(video_bin_counts):
                # Repeat the run_id for 'count' number of timepoints
                group_list.append(np.full(count, run_ids[idx]))
            
            # This is the master group array for this bin_sec
            groups_base = np.concatenate(group_list)

            for normalization in use_normalization:
                fmri_l_proc = fmri_l_base.copy()
                fmri_r_proc = fmri_r_base.copy()
                
                # 3. PERFORM PER-RUN NORMALIZATION OR DEMEANING
                curr_bin_idx = 0 
                for rid in unique_runs:
                    v_indices = timing_df.index[timing_df['run_id'] == rid].tolist()
                    run_bin_length = sum(video_bins[v_indices])
                    run_slice = slice(curr_bin_idx, curr_bin_idx + run_bin_length)
                    
                    if normalization:
                        fmri_l_proc[:, run_slice] = zscore(fmri_l_proc[:, run_slice], axis=1)
                        fmri_r_proc[:, run_slice] = zscore(fmri_r_proc[:, run_slice], axis=1)
                    else:
                        fmri_l_proc[:, run_slice] -= fmri_l_proc[:, run_slice].mean(axis=1, keepdims=True)
                        fmri_r_proc[:, run_slice] -= fmri_r_proc[:, run_slice].mean(axis=1, keepdims=True)
                    
                    curr_bin_idx += run_bin_length
                
                fmri_l_proc = np.nan_to_num(fmri_l_proc)
                fmri_r_proc = np.nan_to_num(fmri_r_proc)
                base_lbl = f"{'normalized' if normalization else 'notnormalized'}{'_hrf' if hrf else ''}"
                if not hrf: base_lbl += f"_delay{int(delay_sec)}s"

                for model_label in models:
                    for mod_name, mod_suffix in modalities.items():
                        
                        file_prefix = model_label.replace('pe-av', 'peav').replace('-', '_').replace('_frame', '-frame')
                        model_path = EMBEDDINGS_BASE / model_label / f"{int(bin_sec)}s" / f"{file_prefix}_{mod_suffix}.npy"
                        current_run_label = f"{model_label}_{mod_name}"
                        if not model_path.exists(): 
                            print(f"  [MISSING] {model_path}")
                            continue
                        
                        # Process Model Embeddings
                        # Note: X shape is (Samples, Features)
                        X = process_model_embeddings(np.load(model_path), bin_sec, timing_df, hrf, normalization)
                        out_dir = OUTPUT_BASE / model_label / base_lbl / f"{int(bin_sec)}s" / mod_name
                        out_dir.mkdir(parents=True, exist_ok=True)
                        
                        # 3. Align Data Lengths
                        # Sometimes minor rounding differences in binning can cause 1-sample mismatches
                        # We truncate to the minimum length to ensure X, Y, and Groups align perfectly.
                        limit = min(X.shape[0], fmri_l_proc.shape[1], groups_base.shape[0])
                        
                        X_trunc = X[:limit]
                        fL_trunc = fmri_l_proc[:, :limit]
                        fR_trunc = fmri_r_proc[:, :limit]
                        groups = groups_base[:limit]
                        
                        print(f"  -> RidgeCV (Group K-Fold): {mod_name} | {base_lbl}")
                        
                        # 4. Run Standard NeuroAI Cross-Validation
                        # We pass 'groups' so the model knows which samples belong to which Run
                        scores_L = run_ridge_cv(X_trunc, fL_trunc, groups, ALPHAS, chunk_size= CHUNK_SIZE)
                        scores_R = run_ridge_cv(X_trunc, fR_trunc, groups, ALPHAS,chunk_size= CHUNK_SIZE)
                        # Use the correlation arrays
                        left_clean = np.nan_to_num(scores_L, nan=0.0)
                        right_clean = np.nan_to_num(scores_R, nan=0.0)
                        
                        # Combine the masked data 
                        combined_data = np.hstack((left_clean, right_clean)).reshape(1, -1)
                        template_img = nib.load(template_cifti_path)
                        
                        if combined_data.shape[1] != template_img.shape[1]:
                            raise ValueError(
                                f"Shape mismatch! Concatenated data: {combined_data.shape[1]} vertices, "
                                f"Template: {template_img.shape[1]}."
                            )

                        cifti = nib.Cifti2Image(
                            combined_data.astype(np.float32), 
                            header=template_img.header,
                            nifti_header=template_img.nifti_header
                        )
                        # Define and update metric name
                        metric_name = "ridge_encoding"
                        try:
                            scalar_axis = cifti.header.get_axis(0)
                            scalar_axis.name[0] = metric_name
                        except Exception as e:
                            print(f"    (Note: Could not update internal map name: {e})")

                        # Fix Output Path (Create dir + add filename)
                        cifti_dir = OUTPUT_BASE / 'cifti_maps'
                        cifti_dir.mkdir(parents=True, exist_ok=True)
                        map_output = cifti_dir / f"ridge_encoding_{current_run_label}_{base_lbl}_{int(bin_sec)}s.dscalar.nii"

                        nib.save(cifti, map_output)
                        print(f"  Saved CIFTI map: {map_output}")

print("\nALL CONFIGURATIONS PROCESSED SUCCESSFULLY!")

STEP 0: LOADING RAW DATA
  Left Data Shape: (32492, 3655) (Expect: Vertices x Time)
  Right Data Shape: (32492, 3655) (Expect: Vertices x Time)

[FMRI] Prep: Bin=1.0s, Delay=0s
Performing Per-Run Normalization...
  -> RidgeCV (Group K-Fold): joint | normalized_hrf
Progress: 29696/29696
Calculating correlations...
Progress: 29716/29716
Calculating correlations...


pixdim[1,2,3] should be non-zero; setting 0 dims to 1


  Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/encoding_results/cifti_maps/ridge_encoding_pe-av-small-16-frame_joint_normalized_hrf_1s.dscalar.nii
  -> RidgeCV (Group K-Fold): joint | notnormalized_hrf
Progress: 29696/29696
Calculating correlations...
Progress: 29716/29716
Calculating correlations...


pixdim[1,2,3] should be non-zero; setting 0 dims to 1


  Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/encoding_results/cifti_maps/ridge_encoding_pe-av-small-16-frame_joint_notnormalized_hrf_1s.dscalar.nii

[FMRI] Prep: Bin=2.0s, Delay=0s
Performing Per-Run Normalization...
  -> RidgeCV (Group K-Fold): joint | normalized_hrf
Progress: 29696/29696
Calculating correlations...
Progress: 29716/29716
Calculating correlations...


...

Progress: 29716/29716
Calculating correlations...
  Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/encoding_results/cifti_maps/ridge_encoding_pe-av-small-16-frame_joint_notnormalized_delay5s_5s.dscalar.nii

[FMRI] Prep: Bin=10.0s, Delay=5.0s
Performing Per-Run Normalization...
  -> RidgeCV (Group K-Fold): joint | normalized_delay5s
Progress: 29696/29696
Calculating correlations...


pixdim[1,2,3] should be non-zero; setting 0 dims to 1


Progress: 29716/29716
Calculating correlations...
  Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/encoding_results/cifti_maps/ridge_encoding_pe-av-small-16-frame_joint_normalized_delay5s_10s.dscalar.nii
  -> RidgeCV (Group K-Fold): joint | notnormalized_delay5s
Progress: 29696/29696
Calculating correlations...


pixdim[1,2,3] should be non-zero; setting 0 dims to 1


Progress: 29716/29716
Calculating correlations...
  Saved CIFTI map: /home/amin/Research/Representation/Movie/outputs/encoding_results/cifti_maps/ridge_encoding_pe-av-small-16-frame_joint_notnormalized_delay5s_10s.dscalar.nii

ALL CONFIGURATIONS PROCESSED SUCCESSFULLY!


In [1]:
def custom_train_test_split(fmri_data, model_data, test_videos, video_ids):
    """
    Custom train-test split where specified videos are considered as test.
    
    Parameters:
    - fmri_data (np.ndarray): The fMRI data (n_samples, n_timepoints).
    - model_data (np.ndarray): The model data (n_samples, n_features).
    - test_videos (list): List of video IDs to use as the test set.
    - video_ids (list): List of all video IDs in the dataset.
    
    Returns:
    - fmri_train (np.ndarray): Training fMRI data.
    - fmri_test (np.ndarray): Test fMRI data.
    - model_train (np.ndarray): Training model data.
    - model_test (np.ndarray): Test model data.
    """
    # Identify indices for the test and train videos
    test_indices = [video_ids.index(vid) for vid in test_videos]
    train_indices = [i for i in range(len(video_ids)) if i not in test_indices]
    
    # Split fMRI data and model data into train and test sets
    fmri_train = fmri_data[:, train_indices]
    fmri_test = fmri_data[:, test_indices]
    model_train = model_data[train_indices, :]
    model_test = model_data[test_indices, :]
    
    return fmri_train, fmri_test, model_train, model_test

In [ ]:
# Convert test_videos to match the format of video_ids (strings)
test_videos = ['video5', 'video9', 'video14', 'video18']  # Use this if video_ids are strings
video_ids = timing_df['Video_ID'].tolist()
# Perform the custom train-test split
fmri_train, fmri_test, model_train, model_test = custom_train_test_split(out_concat, model_demean, test_videos, video_ids)

# Check the shape of train and test sets
print(f"Training fMRI data shape: {fmri_train.shape}")
print(f"Test fMRI data shape: {fmri_test.shape}")
print(f"Training model data shape: {model_train.shape}")
print(f"Test model data shape: {model_test.shape}")

# summaries

In [5]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
from glob import glob
import re

def calculate_top_10_percent(data):
    if len(data) == 0: return 0
    # Sort descending, take top 10%, return mean
    n_top = max(1, int(len(data) * 0.1))
    top_values = np.sort(data)[::-1][:n_top]
    return np.mean(top_values)

def extract_metadata(file_path):
    path_str = file_path.lower()
    parts = file_path.split(os.sep)
    filename = os.path.basename(file_path)
    
    # 1. Locate anchor to find Model and Design
    # Path structure based on notebook: .../encoding_results/{model}/{design}/{bin}/{modality}/{filename}
    try:
        idx = parts.index('encoding_results')
        model_name = parts[idx + 1] 
        design = parts[idx + 2]     
    except (ValueError, IndexError):
        return None # Skip files not in the expected structure

    # 2. Normalization
    # Notebook logic: f"{'normalized' if normalization else 'notnormalized'}..."
    normalization = "Normalized" if "notnormalized" not in design else "Not Normalized"
    
    # 3. HRF status and Delay
    # Notebook logic: ..._{'hrf' if hrf else 'nothrf'}...
    if "nothrf" in design:
        hrf_status = "No"
        # Extract delay from design string (e.g., ..._delay4s)
        delay_match = re.search(r'delay(\d+s?)', design)
        delay = delay_match.group(1) if delay_match else "Unknown"
    else:
        hrf_status = "Yes"
        delay = "N/A"

    # 4. Segment Length / Bin 
    # Notebook logic: f"{int(bin_sec)}s" -> looks for /1s/, /2s/, /10s/
    seg_match = re.search(r'/(\d+s)/', path_str)
    segment_len = seg_match.group(1) if seg_match else "N/A"
    
    # 5. Modality
    modality = "unknown"
    for m in ['joint', 'audio', 'video']:
        if f'/{m}/' in path_str:
            modality = m
            break
            
    # 6. Hemisphere
    # Notebook output: encoding_scores_left.func.gii
    hemi = "L" if "left" in path_str else "R"
    
    return model_name, design, hrf_status, delay, segment_len, normalization, modality, hemi, filename

def generate_encoding_reports(root_dir):
    all_results = []
    
    # Recursive search for gifti files
    files = glob(os.path.join(root_dir, "**/*.func.gii"), recursive=True)
    
    print(f"Found {len(files)} files. Processing...")

    for f in files:
        try:
            meta = extract_metadata(f)
            if meta is None: continue
            
            model, design, hrf, delay, seg, norm, mod, hemi, fname = meta
            
            if mod == "unknown": continue

            # Load GIFTI
            img = nib.load(f)
            data = img.darrays[0].data
            # Filter NaNs/Zeros
            valid_data = data[~np.isnan(data)]
            valid_data = valid_data[valid_data != 0]
            
            if len(valid_data) > 0:
                all_results.append({
                    "Model": model,
                    "Design": design,
                    "HRF": hrf,
                    "Delay": delay,
                    "Segment": seg,
                    "Normalization": norm,
                    "Hemi": hemi,
                    "Avg Score": np.mean(valid_data),
                    "Top 10 Percent": calculate_top_10_percent(valid_data),
                    "Modality": mod,
                    "Filename": fname
                })
        except Exception as e:
            continue

    if not all_results:
        print("No valid encoding result files found.")
        return

    full_df = pd.DataFrame(all_results)
    
    # Define Column Order
    display_cols = ["Model", "Design", "HRF", "Delay", "Segment", "Normalization", "Hemi", "Avg Score", "Top 10 Percent"]

    # Generate 3 Separate Tables
    for mod in ['joint', 'audio', 'video']:
        print("\n" + "="*140)
        print(f"ENCODING REPORT: {mod.upper()} (Sorted by Top 10%)")
        print("="*140)
        
        # Filter by modality
        mod_df = full_df[full_df['Modality'] == mod].copy()
        
        if mod_df.empty:
            print(f"No data found for {mod}")
        else:
            # Sort Descending by Top 10 Percent
            mod_df = mod_df.sort_values(by="Top 10 Percent", ascending=False)
            
            # Print
            print(mod_df[display_cols].to_string(index=False))

# --- EXECUTE ---
# Path derived from your notebook variable: OUTPUT_BASE
base_path = "/home/amin/Research/Representation/Movie/outputs/encoding_results"
generate_encoding_reports(base_path)

Found 198 files. Processing...

ENCODING REPORT: JOINT (Sorted by Top 10%)
               Model                       Design HRF Delay Segment  Normalization Hemi  Avg Score  Top 10 Percent
pe-av-small-16-frame    normalized_nothrf_delay4s  No    4s      5s     Normalized    L   0.352453        0.625772
pe-av-small-16-frame    normalized_nothrf_delay5s  No    5s      5s     Normalized    L   0.356109        0.622138
pe-av-small-16-frame    normalized_nothrf_delay4s  No    4s      5s     Normalized    R   0.355026        0.618142
pe-av-small-16-frame    normalized_nothrf_delay5s  No    5s      5s     Normalized    R   0.357008        0.615279
pe-av-small-16-frame    normalized_nothrf_delay6s  No    6s      5s     Normalized    L   0.339666        0.587099
pe-av-small-16-frame    normalized_nothrf_delay4s  No    4s      2s     Normalized    L   0.316580        0.581976
pe-av-small-16-frame    normalized_nothrf_delay6s  No    6s      5s     Normalized    R   0.339908        0.581452
pe-av